# 01. Data Preprocessing & Validation
This notebook handles data loading, strict sanitation, label verification, duplicate resolution, and text preprocessing variants.

### Step 1: Initial Dataset Loading
We load the raw congressional tweets from `ExtractedTweets.csv` and inspect class ratios.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

path = '/kaggle/input/democratvsrepublicantweets'
target_file = Path(path) / 'ExtractedTweets.csv'

df_raw = pd.read_csv(target_file)
print(f"Raw Data Shape: {df_raw.shape}")
print("\n--- Initial Party Label Counts ---")
print(df_raw['Party'].value_counts(dropna=False))

Raw Data Shape: (86460, 3)

--- Initial Party Label Counts ---
Republican    44392
Democrat      42068
Name: count, dtype: int64


### Step 2: Data Cleaning and Label Normalization
1. Drop missing rows.
2. Ensure only 'Democrat' and 'Republican' labels are kept.
3. Remove identical tweet text strings that carry contradictory labels to avoid class noise.
4. Drop exact duplicate rows (exact matches on Tweet, Party, and Handle).

In [2]:
# Remove nulls
df_clean = df_raw.dropna(subset=['Tweet', 'Party']).copy()

# Filter invalid labels
valid_labels = {'Democrat', 'Republican'}
df_clean = df_clean[df_clean['Party'].isin(valid_labels)]

# Remove cross-party conflicting text duplicate rows
tweet_groups = df_clean.groupby('Tweet')['Party'].nunique()
conflicting_tweets = tweet_groups[tweet_groups > 1].index
df_clean = df_clean[~df_clean['Tweet'].isin(conflicting_tweets)]

# Drop exact duplicates
df_clean = df_clean.drop_duplicates(subset=['Tweet', 'Party', 'Handle'])
print(f"Cleaned Dataset Shape: {df_clean.shape}")

Cleaned Dataset Shape: (86293, 4)


### Step 3: Text Preprocessing Variants
We design three variants to preserve syntax, tenses, and emojis while evaluating the impact of punctuation, URLs, mentions, and hashtags.
- **Variant A**: Lowercasing and whitespace normalization only (preserves punctuation style).
- **Variant B**: Strips URLs, translates punctuation to space (retains words inside hashtags/mentions).
- **Variant C**: Completely removes URLs, mentions (@user), and hashtags (#topic).

In [3]:
import re
import string

df_clean['tweet_original'] = df_clean['Tweet'].copy()

def clean_a(text):
    return re.sub(r'\s+', ' ', str(text)).lower().strip()

def clean_b(text):
    text = str(text).lower()
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    text = re.sub(r'[@#](\w+)', r'\1', text)
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    return re.sub(r'\s+', ' ', text).strip()

def clean_c(text):
    text = str(text).lower()
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()

df_clean['tweet_variant_a'] = df_clean['tweet_original'].apply(clean_a)
df_clean['tweet_variant_b'] = df_clean['tweet_original'].apply(clean_b)
df_clean['tweet_variant_c'] = df_clean['tweet_original'].apply(clean_c)

print("Variants applied!")

Variants applied!


### Step 4: Stratified Splitting without Data Leakage
To prevent text leakage, we drop duplicate text strings globally prior to partitioning into Train, Validation, and Test sets (80/10/10 split) stratified by Party.

In [4]:
from sklearn.model_selection import train_test_split

# Ensure strict text uniqueness to block leakage
df_unique = df_clean.drop_duplicates(subset=['tweet_original']).copy()

train_df, temp_df = train_test_split(
    df_unique, test_size=0.2, random_state=42, stratify=df_unique['Party']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['Party']
)

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

Train: (67564, 7), Val: (8445, 7), Test: (8446, 7)


### Step 5: Export Split Artifacts
We save the preprocessed partition splits locally inside `/content/` for downstream modeling pipelines.

In [5]:
train_df.to_csv('/content/train_split.csv', index=False)
val_df.to_csv('/content/val_split.csv', index=False)
test_df.to_csv('/content/test_split.csv', index=False)
df_clean.to_csv('/content/cleaned_dataset_all_variants.csv', index=False)
print("All splits written to /content/")

All splits written to /content/
